In [1]:
import pandas as pd
import os

In [94]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

data = pd.read_csv(path + "/" + os.listdir(path)[0], encoding='latin-1')

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.


In [95]:
#data = data.sample(frac=.7).reset_index()

In [96]:
data = data[['v1','v2']]
data = data.rename(columns={'v1': 'label', 'v2': 'text'})
data = data.sample(frac=0.85,random_state=13).reset_index(drop=True)


In [97]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
data['label'] = enc.fit_transform(data['label'])
data

,label,text
0,0,"Just looked it up and addie goes back Monday, ..."
1,0,Not..tel software name..
2,0,Lol enjoy role playing much?
3,0,Do we have any spare power supplies
4,0,Neva mind it's ok..
...,...,...
4731,0,"hey, looks like I was wrong and one of the kap..."
4732,0,I'm meeting Darren...
4733,0,Then ur sis how?
4734,1,Congrats! Nokia 3650 video camera phone is you...


In [98]:
model_name = "bert-base-uncased"
MAX_LEN = 128
Batch_size = 32
Epochs = 5
LR = 1e-5
Test_size=0.2
Seed=1
output_dir = "./bert-sentiment"

In [99]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data,test_size=Test_size, random_state=Seed,stratify=data['label'])


In [100]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
  return tokenizer(batch['text'], truncation=True,max_len=128)

In [101]:
train_ds = Dataset.from_pandas(train.reset_index(drop=True))

In [102]:
train_ds = train_ds.map(tokenize,batched=True)

Map:   0%|          | 0/3788 [00:00<?, ? examples/s]

In [103]:
import torch

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=Epochs,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    warmup_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    gradient_accumulation_steps=4,     # simulates batch size of 32
    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_strategy="epoch",      # add this so training loss shows per epoch
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",
    seed=1,
    report_to="none",
)

In [104]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [105]:
test_ds = Dataset.from_pandas(test.reset_index(drop=True))
test_ds = test_ds.map(tokenize,batched=True)

Map:   0%|          | 0/948 [00:00<?, ? examples/s]

In [106]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    data_collator = data_collator,
     callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [107]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.102982,0.058086,0.988397,0.955466
2,0.188195,0.024388,0.993671,0.976923
3,0.086684,0.021154,0.995781,0.984375
4,0.058842,0.020857,0.996835,0.988327
5,0.037393,0.020975,0.996835,0.988327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=300, training_loss=0.29481935342152915, metrics={'train_runtime': 195.7836, 'train_samples_per_second': 96.739, 'train_steps_per_second': 1.532, 'total_flos': 638434864720800.0, 'train_loss': 0.29481935342152915, 'epoch': 5.0})

In [ ]:
y_test = test['label'].to_numpy()
preds = trainer.predict(test_ds)
preds = np.argmax(preds.predictions,axis=-1)

In [ ]:

acc = accuracy_score(y_test,preds)
conf_matrix = confusion_matrix(y_test,preds)
report = classification_report(y_test,preds)
print(report)

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

print(classifier("Hi, this is Dan your"))